In [22]:
# Dates management
import datetime

import numpy as np

import matplotlib.pyplot as plt
# Visualization library
import altair as alt

# Data manipulation library
import pandas as pd

import scipy.stats as stats

# Enable Altair to display all rows of data
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

In [10]:
df_analyse = pd.read_pickle("datap/df_analyse.pkl")
df_analyse.head()

,birth_datetime,death_datetime,gender_source_value,cdm_source,person_id,unique_person_id,visit_occurrence_id,care_site_id,visit_start_datetime,visit_end_datetime,visit_source_value,measurement_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit
0,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,83480500.0,2023-07-09,hb,12.56,g/dL
1,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,84432348.0,2023-07-09,crp,4.64,mg/L
2,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,80623197.0,2023-07-09,bmi,22.70,kg.cm^-2
3,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,83945414.0,2023-07-09,urea,4.05,mmol/L
4,1961-03-29,2021-06-07,female,EHR 1,82132985.0,82132985.0,84984812.0,GHU A.Fleming,2021-06-01,2021-06-07,Hospitalisés,88773683.0,2021-06-01,bmi,26.84,kg.cm^-2


In [ ]:

df_hb = df_analyse[df_analyse['concept_source_value'] == 'hb'].copy()


df_hb['statut_vital'] = df_hb['death_datetime'].isna().map({True: 'En vie', False: 'Décédé'})


chart_vital = alt.Chart(df_hb).mark_boxplot(extent=1.5).encode(
    x=alt.X('statut_vital:N', title='Statut Vital', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('transformed_value:Q', title='Hémoglobine (g/dL)'),
    color=alt.Color('statut_vital:N', legend=None, scale=alt.Scale(scheme='set2'))
).properties(
    width=600,
    height=400,
    title='Distribution de l\'hémoglobine : Décédés vs En vie'
).configure_title(
    fontSize=15,
    anchor='middle'
)

chart_vital.display()   

alt.Chart(...)

In [20]:
df_condition.head()

,visit_occurrence_id,person_id,condition_occurrence_id,condition_source_value
0,85319808.0,84829170.0,87142885.0,T51
1,80902406.0,85929376.0,89378651.0,Z720
2,84295593.0,86868894.0,85796828.0,C504
3,84975826.0,85566948.0,81895083.0,F023
4,83844409.0,89203324.0,87313829.0,Z803


In [21]:
# Jointure à gauche sur les clés patient et visite
df_merged = df_analyse.merge(
    df_condition,
    on=['person_id', 'visit_occurrence_id'],
    how='left'
)

# Affichage des premières lignes pour vérifier l'intégration des nouvelles colonnes
df_merged.head()

,birth_datetime,death_datetime,gender_source_value,cdm_source,person_id,unique_person_id,visit_occurrence_id,care_site_id,visit_start_datetime,visit_end_datetime,visit_source_value,measurement_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit,condition_occurrence_id,condition_source_value
0,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,83480500.0,2023-07-09,hb,12.56,g/dL,82610935.0,G20
1,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,84432348.0,2023-07-09,crp,4.64,mg/L,82610935.0,G20
2,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,80623197.0,2023-07-09,bmi,22.70,kg.cm^-2,82610935.0,G20
3,1984-11-06,NaT,female,EHR 1,88817330.0,88817330.0,84342634.0,Centre F.Sinoussi,2023-07-09,2023-07-27,Hospitalisés,83945414.0,2023-07-09,urea,4.05,mmol/L,82610935.0,G20
4,1961-03-29,2021-06-07,female,EHR 1,82132985.0,82132985.0,84984812.0,GHU A.Fleming,2021-06-01,2021-06-07,Hospitalisés,88773683.0,2021-06-01,bmi,26.84,kg.cm^-2,87304967.0,C504


In [ ]:

#Identifier les diagnostics de cancer (Classification CIM-10 commence par 'C')
df_merged['is_cancer_row'] = df_merged['condition_source_value'].str.startswith('C', na=False)

# soler uniquement les mesures d'hémoglobine
df_hb_merged = df_merged[df_merged['concept_source_value'] == 'hb'].copy()

# On trie de façon décroissante pour que les lignes True (Cancer) soient au-dessus des False
# On supprime les doublons basés sur le même patient et la même visite
df_hb_clean = df_hb_merged.sort_values('is_cancer_row', ascending=False).drop_duplicates(
    subset=['person_id', 'visit_occurrence_id', 'transformed_value'], 
    keep='first'
)

# Création de la variable explicative finale
df_hb_clean['statut_cancer'] = df_hb_clean['is_cancer_row'].map({True: 'Cancer', False: 'Autres / Aucune'})


chart_cancer = alt.Chart(df_hb_clean).mark_boxplot(extent=1.5).encode(
    x=alt.X('statut_cancer:N', title='Diagnostic clinique', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('transformed_value:Q', title='Hémoglobine (g/dL)'),
    color=alt.Color('statut_cancer:N', legend=None, scale=alt.Scale(scheme='set1'))
).properties(
    width=400,
    height=400,
    title='Distribution de l\'hémoglobine : Impact du Cancer'
).configure_title(
    fontSize=15,
    anchor='middle'
)

chart_cancer.display()

# Calcul statistique : Test U de Mann-Whitney
# On extrait les deux vecteurs de valeurs en supprimant les éventuels NaN
groupe_cancer = df_hb_clean[df_hb_clean['is_cancer_row'] == True]['transformed_value'].dropna()
groupe_autre = df_hb_clean[df_hb_clean['is_cancer_row'] == False]['transformed_value'].dropna()

# Exécution du test
u_stat, p_val = stats.mannwhitneyu(groupe_cancer, groupe_autre, alternative='two-sided')

print(f"--- Analyse Statistique ---")
print(f"Effectif Cancer : {len(groupe_cancer)} mesures")
print(f"Effectif Autres : {len(groupe_autre)} mesures")
print(f"Statistique U : {u_stat}")
print(f"Valeur p (p-value) : {p_val:.4e}")

alt.Chart(...)

--- Analyse Statistique ---
Effectif Cancer : 968 mesures
Effectif Autres : 526 mesures
Statistique U : 234060.5
Valeur p (p-value) : 9.9717e-03


In [26]:

#Identifier les diagnostics de cancer (Classification CIM-10 commence par 'C')
df_merged['is_cancer_row'] = df_merged['condition_source_value'].str.startswith('C', na=False)

# Dédoublonner au niveau du PATIENT

df_patient = df_merged.sort_values('is_cancer_row', ascending=False).drop_duplicates(
    subset=['person_id'], 
    keep='first'
).copy()

# Création de la variable explicative
df_patient['statut_cancer'] = df_patient['is_cancer_row'].map({True: 'Cancer', False: 'Autres / Aucune'})

# Calcul de l'âge
df_patient['birth_datetime'] = pd.to_datetime(df_patient['birth_datetime'])
df_patient['visit_start_datetime'] = pd.to_datetime(df_patient['visit_start_datetime'])

df_patient['age'] = (df_patient['visit_start_datetime'] - df_patient['birth_datetime']).dt.days / 365.25
df_patient_clean = df_patient.dropna(subset=['age'])


chart_age_cancer = alt.Chart(df_patient_clean).mark_boxplot(extent=1.5).encode(
    x=alt.X('statut_cancer:N', title='Diagnostic clinique', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('age:Q', title='Âge (années)', scale=alt.Scale(zero=False)),
    color=alt.Color('statut_cancer:N', legend=None, scale=alt.Scale(scheme='set2')) # Changement de palette
).properties(
    width=350,
    height=400,
    title='Distribution de l\'âge : Impact du Cancer'
).configure_title(
    fontSize=15,
    anchor='middle'
)

chart_age_cancer.display()

# Calcul statistique : Test U de Mann-Whitney
groupe_cancer_age = df_patient_clean[df_patient_clean['is_cancer_row'] == True]['age']
groupe_autre_age = df_patient_clean[df_patient_clean['is_cancer_row'] == False]['age']

u_stat_age, p_val_age = stats.mannwhitneyu(groupe_cancer_age, groupe_autre_age, alternative='two-sided')

print(f"--- Analyse Statistique (Âge) ---")
print(f"Effectif Cancer : {len(groupe_cancer_age)} patients")
print(f"Effectif Autres : {len(groupe_autre_age)} patients")
print(f"Statistique U : {u_stat_age}")
print(f"Valeur p (p-value) : {p_val_age:.4e}")

alt.Chart(...)

--- Analyse Statistique (Âge) ---
Effectif Cancer : 968 patients
Effectif Autres : 526 patients
Statistique U : 307768.5
Valeur p (p-value) : 2.4288e-11
